# DSMC Shape Optimisation — Batch Convergence Study

**Goal:** run the star-shape boundary optimisation 30 times with different random seeds 
and check that all runs converge to the same minimum.

**Geometry:**
- Bounding box: `[-5, 5] × [-5, 5]`
- Inscribed square (region of interest): `[-0.5, 0.5] × [-0.5, 0.5]`
- Initial guess: circle of radius `R = 2.0`

**Workflow:**
1. Install dependencies
2. Clone / verify the code
3. Quick smoke test (1 run, 5 iterations)
4. Full 30-run batch
5. Compare & visualise results


## 0 — Configuration

Edit this cell to match your environment before running anything else.

In [ ]:
# ── User-editable settings ─────────────────────────────────────────────────

# How the code was placed on this server:
#   'git'    → git clone from a remote URL (fill REPO_URL below)
#   'upload' → you uploaded / extracted the zip manually (fill REPO_DIR below)
CODE_SOURCE = 'upload'          # 'git' or 'upload'

# If CODE_SOURCE == 'git', set the repository URL:
REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'

# Root directory of the cloned / uploaded repository:
REPO_DIR = '/home/jovyan/dsmc-shape-opt'   # adjust if needed

# Batch settings
N_RUNS    = 30    # number of independent optimisation runs
MAXITER   = 50    # max DE iterations per run
WORKERS   = -1    # -1 = use all CPU cores for the DE population

# Output directories (relative to REPO_DIR)
BATCH_DIR   = 'batch_results'   # where per-run outputs are saved
FIGURES_DIR = 'batch_figures'   # where the comparison figure is saved

print('Settings loaded.')

## 1 — Install Dependencies

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

print('Installing Python packages ...')
pip('numpy', 'scipy', 'matplotlib', 'Pillow')

# gmsh must be installed before pygmsh
pip('gmsh')
pip('pygmsh', 'meshio')

# Optional: animation support
pip('imageio', 'imageio-ffmpeg')

print('All packages installed.')

## 2 — Clone or Verify Code

In [ ]:
import os, sys

if CODE_SOURCE == 'git':
    if not os.path.exists(REPO_DIR):
        print(f'Cloning {REPO_URL} → {REPO_DIR}')
        subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
    else:
        print(f'Directory {REPO_DIR} already exists — pulling latest changes.')
        subprocess.check_call(['git', '-C', REPO_DIR, 'pull'])
elif CODE_SOURCE == 'upload':
    if not os.path.exists(REPO_DIR):
        raise FileNotFoundError(
            f'REPO_DIR="{REPO_DIR}" does not exist.\n'
            'Upload and extract the zip file first, then update REPO_DIR above.'
        )
    print(f'Using existing directory: {REPO_DIR}')

# Add src to the Python path
SRC_DIR = os.path.join(REPO_DIR, 'src')
EXP_DIR = os.path.join(REPO_DIR, 'experiments')
for p in [REPO_DIR, SRC_DIR, EXP_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Quick sanity check
from optimization.config import OPTIMIZATION_CONFIG, SIMULATION_CONFIG, INITIAL_GUESS
print(f'Code loaded OK.')
print(f'  Inscribed square half-side a = {OPTIMIZATION_CONFIG["a"]}')
print(f'  Bounding box half-width    L = {OPTIMIZATION_CONFIG["L"]}')
print(f'  Initial circle radius      R = {INITIAL_GUESS["R"]}')

## 3 — Smoke Test  (1 run × 5 iterations)

Run this cell first to make sure everything works before launching the full batch.

In [ ]:
import os
from optimization.config import OPTIMIZATION_CONFIG, SIMULATION_CONFIG, INITIAL_GUESS
from optimization.run_optimization import run_optimization

smoke_dir = os.path.join(REPO_DIR, 'smoke_test')

opt_cfg = OPTIMIZATION_CONFIG.copy()
opt_cfg['seed']         = 99
opt_cfg['maxiter']      = 5
opt_cfg['workers']      = 1          # single-threaded for smoke test
opt_cfg['save_frames']  = False
opt_cfg['viz_interval'] = 9999
opt_cfg['output_dir']   = smoke_dir

print('Running smoke test (seed=99, 5 iterations) ...')
result, tracker = run_optimization(
    opt_config=opt_cfg,
    sim_params=SIMULATION_CONFIG.copy(),
    initial_guess=INITIAL_GUESS.copy(),
    verbose=True,
)
print(f'\nSmoke test PASSED  —  objective = {result.fun:.6f}')

## 4 — Full Batch  (30 runs)

This cell can take **several hours** depending on the number of CPUs.  
Results are saved incrementally, so you can interrupt and resume with `--resume`.

> **Tip:** run this in a terminal instead of the notebook if you want it to survive
> a browser disconnect:
> ```bash
> cd /home/jovyan/dsmc-shape-opt
> nohup python experiments/batch_run.py --n-runs 30 --workers -1 > batch.log 2>&1 &
> tail -f batch.log
> ```

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(REPO_DIR, 'experiments'))

from batch_run import run_batch

batch_output = os.path.join(REPO_DIR, BATCH_DIR)

batch_summary = run_batch(
    n_runs=N_RUNS,
    maxiter=MAXITER,
    workers=WORKERS,
    output_dir=batch_output,
    resume=False,   # set True to skip already-completed runs
)

print(f"\nDone. {batch_summary['n_successful']}/{N_RUNS} runs succeeded.")

## 5 — Results Comparison

In [ ]:
import os, json
import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(REPO_DIR, 'experiments'))
from compare_results import analyze_batch

batch_output   = os.path.join(REPO_DIR, BATCH_DIR)
figures_output = os.path.join(REPO_DIR, FIGURES_DIR)

analyze_batch(
    results_dir=batch_output,
    output_dir=figures_output,
    convergence_pct=5.0,
)

# Display the figure inline
from IPython.display import Image, display
fig_path = os.path.join(figures_output, 'batch_analysis.png')
if os.path.exists(fig_path):
    display(Image(fig_path))
else:
    print(f'Figure not found at {fig_path}')

## 6 — Quick Statistics Printout

In [ ]:
import json, os
import numpy as np

summary_path = os.path.join(REPO_DIR, BATCH_DIR, 'batch_summary.json')
with open(summary_path) as f:
    s = json.load(f)

successful = [r for r in s['runs'] if r.get('success') and 'objective' in r]
objs = np.array([r['objective'] for r in successful])

print(f"Runs completed : {len(s['runs'])} / {s['n_runs']}")
print(f"Successful     : {len(successful)}")
print()
print(f"Min  objective : {objs.min():.8f}")
print(f"Mean objective : {objs.mean():.8f}")
print(f"Std  objective : {objs.std():.8f}")
print(f"Max  objective : {objs.max():.8f}")
print()

threshold = objs.min() * 1.05
converged = (objs <= threshold).sum()
print(f"Converged to within 5% of min: {converged} / {len(objs)}")

if 'best_run' in s:
    br = s['best_run']
    print(f"\nBest run : run_{br['run_id']:02d}_seed{br['seed']}")
    print(f"Best obj : {br['objective']:.8f}")
    print(f"Best c   : {np.round(br['coefficients'], 5)}")

## 7 — (Optional) Run a Single Best-Quality Final Simulation

Re-run the best coefficients with high particle count for publication-quality results.

In [ ]:
import json, os
import numpy as np
from optimization.shape_optimizer import evaluate_with_details
from optimization.config import OPTIMIZATION_CONFIG, SIMULATION_CONFIG
from optimization.viz_optimizer import visualize_iteration, OptimizationTracker

summary_path = os.path.join(REPO_DIR, BATCH_DIR, 'batch_summary.json')
with open(summary_path) as f:
    s = json.load(f)

best_c = np.array(s['best_run']['coefficients'])

# High-resolution simulation config
hq_sim = SIMULATION_CONFIG.copy()
hq_sim['N']      = 10000   # more particles
hq_sim['n_tot']  = 200     # more time steps

hq_dir = os.path.join(REPO_DIR, 'hq_final')
os.makedirs(hq_dir, exist_ok=True)

opt_cfg = OPTIMIZATION_CONFIG.copy()
opt_cfg['output_dir'] = hq_dir

print('Running high-quality final simulation on best shape ...')
results = evaluate_with_details(best_c, hq_sim, opt_cfg)

if results:
    tracker = OptimizationTracker(output_dir=hq_dir)
    visualize_iteration(best_c, results, opt_cfg, iteration=0, tracker=tracker)
    from IPython.display import Image, display
    import glob
    frames = sorted(glob.glob(os.path.join(hq_dir, 'frames', '*.png')))
    if frames:
        display(Image(frames[-1]))
    print(f'KE in square   : {results["ke_in_square"]:.6f}')
    print(f'KE total       : {results["ke_total"]:.6f}')
    print(f'Particles in □ : {results["num_particles_in_square"]} / {results["num_particles_total"]}')
else:
    print('Simulation failed.')